# 02 · SQL on DuckDB: the warehouse and the lake

The same engine answers questions from the star schema (a DuckDB file) and directly from Parquet.

In [ ]:
# Run from the repository root (`make notebooks` does). Requires `make demo` first.
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import pandas as pd
import duckdb
pd.set_option("display.width", 140)

In [ ]:
wh = duckdb.connect("data/warehouse/finflow.duckdb", read_only=True)
wh.execute("SET integer_division = true")
wh.sql("SHOW TABLES").df()

## Star schema: daily volume and failure rate

In [ ]:
wh.sql("SELECT * FROM mart_daily_transactions ORDER BY full_date").df().head(10)

## Window functions: 7-day rolling average

See `sql/examples/11_running_totals_and_rolling_averages.sql` for the commented version.

In [ ]:
wh.sql(Path("sql/examples/11_running_totals_and_rolling_averages.sql").read_text()).df().head(10)

## SCD Type 2: one customer, several versions

In [ ]:
wh.sql("""
SELECT customer_id, segment, city, valid_from, valid_to, is_current
FROM dim_customer
WHERE customer_id IN (SELECT customer_id FROM dim_customer GROUP BY 1 HAVING COUNT(*) > 1)
ORDER BY customer_id, valid_from
""").df()

## The lake, without a warehouse

In [ ]:
lake = duckdb.connect()
lake.sql("""
SELECT txn_date, COUNT(*) AS n
FROM read_parquet('data/lake/silver/transactions/*/*.parquet', hive_partitioning = true)
GROUP BY txn_date ORDER BY txn_date
""").df().head()

In [ ]:
# EXPLAIN shows that only the matching partition directory is read.
print(lake.sql("""
EXPLAIN SELECT COUNT(*) FROM read_parquet('data/lake/silver/transactions/*/*.parquet', hive_partitioning = true)
WHERE txn_date = DATE '2026-01-15'
""").df().iloc[0, 1])